# 02 - Experience A : decalage de domaine (SRM)

Hypothese H1 : un detecteur entraine sur des images naturelles perd sa fiabilite sur des images
generees. On mesure l'AUC de reference, la chute en cross-domaine, l'AUC appariee et le taux de faux
positifs, pour chaque algorithme d'insertion.

On travaille sur les caracteristiques SRM, deja extraites et deposees sur Drive. Les figures sont
enregistrees selon la nomenclature expA_srm_<type>_<algo>_p<charge>.png.

## Configuration

In [ ]:
# ================= CONFIGURATION =================
FEATURE  = 'srm'
ALGOS    = ['lsb', 'uniward', 'hill']   # algorithmes a analyser
PAYLOAD  = 0.4
GENERES  = ['sd', 'sdxl', 'adm']        # sources generees comparees au naturel
N_PCA    = 300                          # composantes gardees pour reduire SRM
SEED     = 42
# ================================================
pp = str(PAYLOAD).replace('.', '')      # pour nommer les figures, 0.4 -> 04
print('Experience A sur', FEATURE, '| algos', ALGOS, '| charge', PAYLOAD)

In [ ]:
import os, glob
import numpy as np
np.random.seed(SEED)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')
FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
FIG = f'{DATA_DIR}/figures'
os.makedirs(FIG, exist_ok=True)
print('Caracteristiques :', FEAT_DIR)
print('Figures :', FIG)

In [ ]:
!pip install -q scikit-learn matplotlib
print('Installation terminee.')

## 1. Chargement et detecteur

SRM ayant des dizaines de milliers de dimensions, le detecteur reduit d'abord par PCA, puis classe par
regression logistique. Sans cette reduction, le modele surapprend.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

def charger(source, setname):
    return np.load(f'{FEAT_DIR}/{source}__{setname}.npy')

def jeu(source, algo):
    Xc = charger(source, 'cover')
    Xs = charger(source, f'{algo}_p{PAYLOAD}')
    n = min(len(Xc), len(Xs)); Xc, Xs = Xc[:n], Xs[:n]
    X = np.vstack([Xc, Xs]); y = np.concatenate([np.zeros(n), np.ones(n)])
    return X, y, Xc

def detecteur():
    return make_pipeline(StandardScaler(),
                         PCA(n_components=N_PCA, random_state=SEED),
                         LogisticRegression(max_iter=5000))

def auc_croisee(X, y):
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    s = cross_val_score(detecteur(), X, y, cv=cv, scoring='roc_auc')
    return s.mean(), s.std()

print('Detecteur SRM pret, PCA a', N_PCA, 'composantes.')

## 2. Analyse et figures par algorithme

Pour chaque algorithme, on calcule les trois mesures et on produit trois figures : les courbes ROC, les
barres d'AUC, et la distribution cumulee des scores sur images vierges qui illustre les faux positifs.

In [ ]:
import matplotlib.pyplot as plt

resume_global = {}
for algo in ALGOS:
    Xn, yn, cover_nat = jeu('natural', algo)
    auc_ref_m, auc_ref_s = auc_croisee(Xn, yn)
    det_nat = detecteur().fit(Xn, yn)
    seuil = np.quantile(det_nat.predict_proba(cover_nat)[:, 1], 0.95)

    lignes = []
    for g in GENERES:
        Xg, yg, cover_g = jeu(g, algo)
        auc_cross = roc_auc_score(yg, det_nat.predict_proba(Xg)[:, 1])
        am, asd = auc_croisee(Xg, yg)
        fpr_g = float(np.mean(det_nat.predict_proba(cover_g)[:, 1] > seuil))
        lignes.append((g, auc_cross, am, asd, fpr_g))
    resume_global[algo] = (auc_ref_m, auc_ref_s, lignes)

    print(f'\n=== {algo} {PAYLOAD} bpp ===')
    print(f'reference naturel AUC {auc_ref_m:.3f} +/- {auc_ref_s:.3f}')
    for g, ac, am, asd, fp in lignes:
        print(f'  {g:5s} cross {ac:.3f}  apparie {am:.3f}  FPR {fp:.3f}')

    # Figure 1 : courbes ROC
    fig, ax = plt.subplots(figsize=(7, 6))
    Xtr, Xte, ytr, yte = train_test_split(Xn, yn, test_size=0.3, random_state=SEED, stratify=yn)
    det_ref = detecteur().fit(Xtr, ytr)
    f, t, _ = roc_curve(yte, det_ref.predict_proba(Xte)[:, 1])
    ax.plot(f, t, lw=2, label=f'reference (AUC={roc_auc_score(yte, det_ref.predict_proba(Xte)[:,1]):.2f})')
    for g in GENERES:
        Xg, yg, _ = jeu(g, algo)
        sc = det_nat.predict_proba(Xg)[:, 1]
        f, t, _ = roc_curve(yg, sc)
        ax.plot(f, t, label=f'cross vers {g} (AUC={roc_auc_score(yg, sc):.2f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_xlabel('faux positifs'); ax.set_ylabel('vrais positifs')
    ax.set_title(f'ROC, {algo} {PAYLOAD} bpp, SRM'); ax.legend()
    nom = f'{FIG}/expA_srm_roc_{algo}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

    # Figure 2 : barres d'AUC
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = ['ref'] + [f'{g}\ncross' for g in GENERES] + [f'{g}\napparie' for g in GENERES]
    vals = [auc_ref_m] + [l[1] for l in lignes] + [l[2] for l in lignes]
    couleurs = ['#2a9d8f'] + ['#e76f51'] * len(GENERES) + ['#264653'] * len(GENERES)
    ax.bar(labels, vals, color=couleurs); ax.axhline(0.5, ls='--', color='gray')
    ax.set_ylim(0.4, 1.0); ax.set_title(f'AUC par condition, {algo} {PAYLOAD} bpp, SRM')
    nom = f'{FIG}/expA_srm_aucbars_{algo}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

    # Figure 3 : distribution cumulee des scores sur images vierges
    fig, ax = plt.subplots(figsize=(9, 5))
    for s, c in [('natural', 'tab:blue'), ('sd', 'tab:orange'), ('sdxl', 'tab:green'), ('adm', 'tab:red')]:
        cov = charger(s, 'cover')
        sc = det_nat.predict_proba(cov)[:, 1]; xs = np.sort(sc)
        ax.plot(xs, np.arange(1, len(xs) + 1) / len(xs), color=c, lw=2, label=f'{s} vierge')
    ax.axvline(seuil, ls='--', color='gray', label='seuil 5% FPR naturel')
    ax.set_xlabel('score du detecteur'); ax.set_ylabel('proportion cumulee')
    ax.set_title(f'Scores sur images vierges, {algo} {PAYLOAD} bpp, SRM'); ax.legend()
    nom = f'{FIG}/expA_srm_fprcdf_{algo}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

## Suite

Les figures sont dans le dossier figures de Drive, nommees selon la nomenclature. On les copie dans le
dossier results du depot, et on renseigne leur interpretation dans results/figures_interpretations.md,
chaque figure etant referencee par son nom.